# 🥏 Ground frisbee detection with classical CV

Part 1: no learned model. Threshold RGB brightness sum and channel differences, then fit ellipses to the cleaned binary mask.

**Prerequisite:** in the project folder, run `pip install -r requirements.txt jupyterlab`.


In [ ]:
from pathlib import Path
import cv2
import numpy as np
from PIL import Image
from IPython.display import display
from detector import DetectorSettings, detect, draw_detections, prepare_frame

ROOT = Path.cwd()
print("Working directory:", ROOT)


## 1. Choose an example and tune white segmentation

Change the values below and re-run the cells to tune the white mask.


In [ ]:
image_path = ROOT / "samples" / "02_angled_white_disc.png"

settings = DetectorSettings(
    rgb_sum_min=435,
    channel_difference_max=30,
    min_radius=12,
    max_radius=120,
)

frame = cv2.imread(str(image_path))
assert frame is not None, f"Cannot read {image_path}"
frame = prepare_frame(frame)
display(Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))


## 2. Run detection and see the result

The match score is a simple heuristic, **not a probability** that an object is a frisbee.


In [ ]:
detections, debug = detect(frame, settings)
annotated = draw_detections(frame, detections, settings)
print("Ground frisbee-like candidate found:", bool(detections))
for i, candidate in enumerate(detections, 1):
    print(i, candidate)
display(Image.fromarray(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)))


## 3. Inspect the RGB threshold mask and cleaned detection mask


In [ ]:
print("RGB threshold mask:")
display(Image.fromarray(debug["threshold_mask"]))
print("Cleaned detection mask:")
display(Image.fromarray(debug["white_mask"]))


## 4. Compare strict and relaxed channel differences


In [ ]:
from dataclasses import replace

for difference_limit in [15, 30, 60]:
    config = replace(settings, channel_difference_max=difference_limit)
    found, debug = detect(frame, config)
    print(f"Maximum channel difference {difference_limit}: {len(found)} candidates")
    display(Image.fromarray(debug["white_mask"]))
    display(Image.fromarray(cv2.cvtColor(draw_detections(frame, found, config), cv2.COLOR_BGR2RGB)))


## Next: real camera frames

Place a still frame from your camera in this folder, then change `image_path` above. The detector searches the entire image.

For an interactive web app run `python -m streamlit run website/app.py`. For continuous video run `python -m camera.webcam --input 0`.
